<a href="https://colab.research.google.com/github/taichi0520-hub/my-research-project/blob/main/src/T_junction_simulation_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1次Markov過程モデルに基づいたT字分岐のシミュレーションコード(グラフ作成まで)


### 使用方法


1.   1番上のコードにおける「#パラメータ設定」の部分の値を適宜変更して実行してください。それ以外の部分で記載されているn_cellsなどの値は、パラメータ設定の部分で明示的に指定されなかった場合のデフォルト値のため変更しなくて大丈夫です。

2.   コードの下部に「グラフを保存しました」と出力されたら、画面左のファイル(Google drive)の中にグラフの画像ファイルがPNG形式で2種類保存されているはずです。

3.   下の2つ目のコードを実行し、シミュレーション結果と解析結果をまとめたExcelファイルを出力してください。



*再度実行しても以前保存したファイルが上書きされないように、日時(yyyymmdd_hhmmss)をファイル名に入れ、ファイル名が重複しないようにしています。



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from datetime import datetime
import matplotlib.ticker as ticker

def simulate_t_junction(n_cells=20, alpha=0.0):
    p_geo = 0.5
    results = [random.randint(0, 1)] #先頭細胞
    p_same = p_geo + alpha
    for i in range(1, n_cells):
        prev = results[-1]
        results.append(prev if random.random() < p_same else 1 - prev)
    return results

def get_run_lengths(seq):
    runs = []
    current_run = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i-1]:
            current_run += 1
        else:
            runs.append(current_run)
            current_run = 1
    runs.append(current_run)
    return runs

def get_stats_for_alpha(alpha, n_trials=20, n_cells=20):
    left_counts = []
    trial_runs_list = []
    max_run = 0

    for _ in range(n_trials):
        seq = simulate_t_junction(n_cells, alpha)
        left_counts.append(seq.count(0)) # 0を左(L)としてカウント
        runs = get_run_lengths(seq)
        trial_runs_list.append(runs)
        if runs:
            max_run = max(max_run, max(runs))

    # 右(R)の数は (総数 - 左の数)
    right_counts = [n_cells - l for l in left_counts]

    # 各トライアルごとのRun lengthの確率（frequency）を計算して平均と標準偏差を出す
    freqs_matrix = np.zeros((n_trials, max_run))
    for t, runs in enumerate(trial_runs_list):
        for r in runs:
            freqs_matrix[t, r-1] += 1
        if len(runs) > 0:
            freqs_matrix[t, :] /= len(runs)

    mean_freqs = np.mean(freqs_matrix, axis=0)
    std_freqs = np.std(freqs_matrix, axis=0)

    # 平均と標準偏差を計算
    return {
        'mean_L': np.mean(left_counts),
        'std_L': np.std(left_counts),
        'mean_R': np.mean(right_counts),
        'std_R': np.std(right_counts),
        'run_lengths': list(range(1, max_run + 1)),
        'mean_freqs': mean_freqs,
        'std_freqs': std_freqs
    }

# パラメータ設定
alphas = [0.0, 0.2, -0.2]
labels = ['Independent (α=0)', 'Attraction (α=0.2)', 'Repulsion (α=-0.2)']
markers = ['o', 's', '^'] # αごとにマーカーの形を変える

#この部分のみ値を変更すればOK
N_CELLS = 20 #細胞数
N_TRIALS = 20 #試行回数

# プロットの準備 (2つの別々のfigureを作成)
fig1, ax1 = plt.subplots(figsize=(8, 6))
fig2, ax2 = plt.subplots(figsize=(8, 6))
x = np.arange(len(alphas))
width = 0.35

for i, alpha in enumerate(alphas):
    stats = get_stats_for_alpha(alpha, n_trials=N_TRIALS, n_cells=N_CELLS)

    # グラフ1: 左右の細胞数分布
    # 左(L)の棒とエラーバー
    ax1.bar(i - width/2, stats['mean_L'], width, yerr=stats['std_L'],
           capsize=5, label='Left' if i==0 else "", color='skyblue', edgecolor='black')
    # 右(R)の棒とエラーバー
    ax1.bar(i + width/2, stats['mean_R'], width, yerr=stats['std_R'],
           capsize=5, label='Right' if i==0 else "", color='salmon', edgecolor='black')

    # グラフ2: Run lengthの分布 (凡例にエラーバーを含めないため、plotとerrorbarを分ける)
    p = ax2.plot(stats['run_lengths'], stats['mean_freqs'], marker=markers[i], label=labels[i])

    # エラーバーの下限が0を下回らないように調整
    lower_err = np.minimum(stats['mean_freqs'], stats['std_freqs'])
    upper_err = stats['std_freqs']

    ax2.errorbar(stats['run_lengths'], stats['mean_freqs'], yerr=[lower_err, upper_err],
                 fmt='none', capsize=4, ecolor=p[0].get_color())

info_text = f"Cells = {N_CELLS}, Trials = {N_TRIALS}"

# グラフ1の設定
ax1.set_ylabel('Mean Cell Count')
ax1.yaxis.set_major_locator(ticker.MaxNLocator(integer=True)) # 縦軸を整数のみに設定
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.legend(frameon=False, loc='upper right', bbox_to_anchor=(1.0, 0.92))
ax1.tick_params(direction='in') # 目盛りを内向きに
ax1.text(0.98, 0.98, info_text, transform=ax1.transAxes, ha='right', va='top')
ax1.spines['top'].set_visible(False) # 上の枠線を消す
ax1.spines['right'].set_visible(False) # 右の枠線を消す

# グラフ2の設定
ax2.set_xlabel('Run length')
ax2.set_ylabel('Frequency')
ax2.set_ylim(bottom=0) # 縦軸の下限を0に
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True)) # 横軸を整数のみに設定
ax2.legend(frameon=False, loc='upper right', bbox_to_anchor=(1.0, 0.92))
ax2.tick_params(direction='in') # 目盛りを内向きに
ax2.text(0.98, 0.98, info_text, transform=ax2.transAxes, ha='right', va='top')
ax2.spines['top'].set_visible(False) # 上の枠線を消す
ax2.spines['right'].set_visible(False) # 右の枠線を消す

# グラフの軸の上限を目盛り線で終わらせる
fig1.canvas.draw()
ax1.set_ylim(0, ax1.get_yticks()[-1])

fig2.canvas.draw()
ax2_xticks = ax2.get_xticks()
ax2.set_xticks(ax2_xticks) # 目盛りを固定
ax2.set_xlim(ax2_xticks[0], ax2_xticks[-1]) # 固定された目盛りの端をxlimに設定
ax2_yticks = ax2.get_yticks()
ax2.set_yticks(ax2_yticks) # 目盛りを固定
ax2.set_ylim(0, ax2_yticks[-1]) # 固定された目盛りの端をylimに設定

# タイムスタンプを使って一意のファイル名を生成
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename1 = f'cell_distribution_{timestamp}.png'
filename2 = f'run_lengths_{timestamp}.png'

# レイアウト調整と保存
fig1.tight_layout()
fig1.savefig(filename1)

fig2.tight_layout()
fig2.savefig(filename2)

# 表示
plt.show()

print(f"グラフを保存しました:\n - {filename1}\n - {filename2}")

In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np

# 出力用のリスト
csv_data = []

for alpha in alphas:
    left_counts = []
    right_counts = []

    # 1. まず各αにおける全試行のデータを収集する
    alpha_trials = []
    for trial in range(N_TRIALS):
        seq = simulate_t_junction(N_CELLS, alpha)
        # 0をL, 1をRとして文字列に変換
        seq_str = "".join(["L" if s == 0 else "R" for s in seq])
        l_count = seq.count(0)
        r_count = N_CELLS - l_count
        runs = get_run_lengths(seq)
        mean_cluster = np.mean(runs) if runs else 0

        left_counts.append(l_count)
        right_counts.append(r_count)

        alpha_trials.append({
            'Alpha (α)': alpha,
            'Trial': trial + 1,
            'Sequence (LR)': seq_str,
            'L_Count': l_count,
            'R_Count': r_count,
            'Run_Lengths': str(runs),
            'Mean_Cluster_Size': mean_cluster
        })

    # 2. αごとの全体的な統計量（平均や標準偏差）を計算
    std_L = np.std(left_counts)
    std_R = np.std(right_counts)
    mean_L = np.mean(left_counts)
    mean_R = np.mean(right_counts)

    # 3. 各試行のデータに全体の統計量を結合
    for trial_data in alpha_trials:
        trial_data['Alpha_Mean_L'] = mean_L
        trial_data['Alpha_Std_L'] = std_L
        trial_data['Alpha_Mean_R'] = mean_R
        trial_data['Alpha_Std_R'] = std_R
        csv_data.append(trial_data)

# データフレームへの変換
df = pd.DataFrame(csv_data)

# 項目の説明データフレーム
explanation_data = [
    {"項目名": "Parameter", "説明": "設定項目名（N_CELLS: 細胞数、N_TRIALS: 試行回数など）"},
    {"項目名": "Value", "説明": "各設定項目に割り当てられた値"},
    {"項目名": "Alpha (α)", "説明": "細胞が前回と同じ方向に進む傾向の強さ（相互作用パラメータ―）"},
    {"項目名": "Trial", "説明": "シミュレーション試行の識別番号"},
    {"項目名": "Sequence (LR)", "説明": "各試行でシミュレートされた細胞の分岐方向の並び（L: 左, R: 右）"},
    {"項目名": "L_Count", "説明": "その試行において、左（L）に分岐した細胞の総数"},
    {"項目名": "R_Count", "説明": "その試行において、右（R）に分岐した細胞の総数"},
    {"項目名": "Run_Lengths", "説明": "同じ方向に連続して分岐した長さのリスト"},
    {"項目名": "Mean_Cluster_Size", "説明": "その試行におけるランレングスの平均値（平均クラスターサイズ）"},
    {"項目名": "Alpha_Mean_L", "説明": "現在のα条件における、全試行でのL_Countの平均値"},
    {"項目名": "Alpha_Std_L", "説明": "現在のα条件における、全試行でのL_Countの標準偏差"},
    {"項目名": "Alpha_Mean_R", "説明": "現在のα条件における、全試行でのR_Countの平均値"},
    {"項目名": "Alpha_Std_R", "説明": "現在のα条件における、全試行でのR_Countの標準偏差"}
]
explanation_df = pd.DataFrame(explanation_data)

# タイムスタンプ付きのファイル名を生成
timestamp_file = datetime.now().strftime("%Y%m%d_%H%M%S")
excel_filename = f'simulation_results_{timestamp_file}.xlsx'

# Excelファイルとして複数シートに分けて出力
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    # 1シート目: 項目の説明
    explanation_df.to_excel(writer, sheet_name='項目の説明', index=False)

    # 2シート目: パラメータ設定
    params_df = pd.DataFrame({
        'Parameter': ['N_CELLS', 'N_TRIALS'],
        'Value': [N_CELLS, N_TRIALS]
    })
    params_df.to_excel(writer, sheet_name='Parameters', index=False)

    # αの値ごとに別シートに出力
    for alpha in df['Alpha (α)'].unique():
        alpha_df = df[df['Alpha (α)'] == alpha]
        sheet_name = f'Alpha_{alpha}'
        alpha_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Excelファイルを保存しました:\n - {excel_filename}\n")

# 中身の確認 (最初の数行を表示)
display(df.head())